In [58]:
!git pull

There is no tracking information for the current branch.
Please specify which branch you want to merge with.
See git-pull(1) for details.

    git pull <remote> <branch>

If you wish to set tracking information for this branch you can do so with:

    git branch --set-upstream-to=origin/<branch> main



In [60]:
!git remote set-url origin https://github.com/sparkyyyhd-bu/bu-rise-music.git

In [61]:
!git fetch origin
!git checkout audio-based

remote: Enumerating objects: 947, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (183/183), done.
remote: Total 947 (delta 199), reused 149 (delta 85), pack-reused 678 (from 1)
Receiving objects: 100% (947/947), 59.21 MiB | 4.63 MiB/s, done.
Resolving deltas: 100% (623/623), done.
From https://github.com/sparkyyyhd-bu/bu-rise-music
 * [new branch]      audio-based        -> origin/audio-based
 * [new branch]      main               -> origin/main
 * [new branch]      playlist-gen-proto -> origin/playlist-gen-proto
branch 'audio-based' set up to track 'origin/audio-based'.
Switched to a new branch 'audio-based'


In [62]:
from pathlib import Path

# Verify data_utils.py is now present
data_utils_path = Path("src/training/data_utils.py")
print("File exists:", data_utils_path.exists())

File exists: False


In [63]:
import sys
from pathlib import Path

# 1. Search for the file starting from your current notebook location
repo_root = Path.cwd().resolve()
while not (repo_root / "src" / "training" / "data_utils.py").is_file():
    repo_root = repo_root.parent
    if str(repo_root) == repo_root.anchor:
        raise FileNotFoundError("Could not find data_utils.py! Ensure you pulled the audio-based branch.")

# 2. Add it to Python's path and import
sys.path.insert(0, str(repo_root / "src"))
from training.data_utils import grouped_track_id_split

print("Successfully found and imported data_utils.py from:", repo_root)

Successfully found and imported data_utils.py from: /Users/eric/Downloads


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



In [22]:
from sklearn.model_selection import GridSearchCV

In [3]:
from sklearn.ensemble import GradientBoostingRegressor

In [4]:
from sklearn.linear_model import LinearRegression

In [5]:
from sklearn.neural_network import MLPRegressor

In [6]:
from sklearn.svm import SVR

In [7]:
from catboost import CatBoostRegressor

In [8]:
from sklearn.model_selection import RandomizedSearchCV

In [9]:
import re

In [65]:
df = pd.read_csv("spotify_tracks.csv")
track_ids = df["id"].astype(str).tolist()

In [68]:
df1 = pd.read_csv("spotify_tracks.csv")
df2 = pd.read_csv("lyrics_features.csv")

In [69]:
df2_clean = df2[df2['mean_syllables_word'] != -1].copy()

# 2. Extract 'id', 'popularity', AND 'lyrics' from df1
df1_extracted = df1[['id', 'popularity', 'lyrics']].copy()

# 3. Clean the 'lyrics' column in df1_extracted before merging
def clean_lyric_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    # Remove carriage returns (\r) and newlines (\n) with a space
    text = re.sub(r'[\r\n]+', ' ', text)
    # Collapse multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text)
    # Strip leading and trailing whitespace
    return text.strip()

df1_extracted['lyrics'] = df1_extracted['lyrics'].apply(clean_lyric_text)

# 4. Merge df2_clean with df1_extracted on matching track IDs
df = pd.merge(
    df2_clean, 
    df1_extracted, 
    left_on='track_id', 
    right_on='id', 
    how='inner'
)

# 5. Drop redundant 'id' and 'Unnamed' index columns
df = df.drop(columns=['id'])
columns_to_drop = [col for col in df.columns if 'Unnamed' in col]
df = df.drop(columns=columns_to_drop)

# Verify the lyrics are clean and present
print(df[['track_id', 'popularity', 'lyrics']].head())

                 track_id  popularity  \
0  13keyz9ikBe6ZpRasw7l4X        52.0   
1  1WugzepXsLjnsM0K4UaWYc        55.0   
2  2MO6oEAlMKcsfI8xP3yoy8        46.0   
3  1i4St7fmSUE9nB3R9n8fol        36.0   
4  3UyfvY3Gs6d4wvq8O4ANqQ         6.0   

                                              lyrics  
0  It was one of those times what a real good tim...  
1  Ain't' a lifestyle that I would rather Look in...  
2  Tú, tú eres lo que yo escogí Yo quiero perderm...  
3  Now I've had the time of my life No, I never f...  
4  On est pas là pour payer les dettes On a tous ...  


In [70]:
df['word_density'] = df['n_words'] / (df['n_sentences'] + 1e-5)

In [71]:
from langdetect import detect, DetectorFactory
import numpy as np

DetectorFactory.seed = 0

def get_lang(text):
    try:
        if len(str(text).strip()) < 10:
            return 'unknown'
        return detect(text)
    except:
        return 'unknown'

# 1. Detect language
df['lang'] = df['lyrics'].apply(get_lang)

# 2. Binary indicator for English
df['is_english'] = (df['lang'] == 'en').astype(int)

# 3. For non-English songs, set sentiment to NaN or 0, 
# but the model now knows 'is_english = 0' so it won't misinterpret 0.0 as neutral English sentiment!
df['sentiment_polarity'] = df['lyrics'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df['sentiment_polarity'] = np.where(df['is_english'] == 1, df['sentiment_polarity'], 0)

# Add 'is_english' directly into your feature list


In [72]:


# 1. Type-Token Ratio (Vocabulary Diversity)
def get_ttr(text):
    words = str(text).lower().split()
    return len(set(words)) / len(words) if len(words) > 0 else 0

# 2. Repetitiveness Score (Popular tracks repeat hooks often!)
def get_repetition(text):
    words = str(text).lower().split()
    if len(words) == 0:
        return 0
    return 1.0 - (len(set(words)) / len(words))

# 3. Average Word Length (Character count per word)
def get_avg_word_length(text):
    words = str(text).split()
    if not words:
        return 0
    return sum(len(w) for w in words) / len(words)

# Apply to DataFrame
df['ttr'] = df['lyrics'].apply(get_ttr)
df['repetition_score'] = df['lyrics'].apply(get_repetition)
df['avg_word_length'] = df['lyrics'].apply(get_avg_word_length)

In [73]:
def extract_advanced_lyric_features(text):
    lines = [line.strip() for line in str(text).split('\n') if line.strip()]
    if not lines:
        return {
            'syllables_per_line': 0,
            'syllables_per_word': 0,
            'syllable_variation': 0,
            'novel_word_prop': 0
        }
    
    # Simple heuristic syllable counter (vowel group count per word)
    def count_syllables(word):
        word = word.lower()
        count = len(re.findall(r'[aeiouy]+', word))
        return max(1, count)

    line_syllables = []
    total_words = 0
    total_syllables = 0
    novel_props = []

    for i, line in enumerate(lines):
        words = re.findall(r'\b\w+\b', line.lower())
        n_words = len(words)
        n_syls = sum(count_syllables(w) for w in words) if words else 0
        
        line_syllables.append(n_syls)
        total_words += n_words
        total_syllables += n_syls

        # Novel Word Proportion between line pair (i-1) and line (i)
        if i > 0:
            prev_words = set(re.findall(r'\b\w+\b', lines[i-1].lower()))
            if words:
                new_words = [w for w in words if w not in prev_words]
                novel_props.append(len(new_words) / len(words))

    avg_syl_line = np.mean(line_syllables) if line_syllables else 0
    avg_syl_word = (total_syllables / total_words) if total_words > 0 else 0
    syl_var = np.std(line_syllables) if len(line_syllables) > 1 else 0
    avg_novel_prop = np.mean(novel_props) if novel_props else 0

    return {
        'syllables_per_line': avg_syl_line,
        'syllables_per_word': avg_syl_word,
        'syllable_variation': syl_var,
        'novel_word_prop': avg_novel_prop
    }

# Apply fast extractor across full dataset
features_df = df['lyrics'].apply(extract_advanced_lyric_features).apply(pd.Series)
df = pd.concat([df, features_df], axis=1)

In [74]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# 1. Convert lyrics into top word patterns
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', min_df=5)
X_tfidf = tfidf.fit_transform(df['lyrics'])

# 2. Compress down to 15 key semantic components
svd = TruncatedSVD(n_components=15, random_state=10)
X_svd = svd.fit_transform(X_tfidf)

# 3. Create DataFrame for SVD features
svd_cols = [f'svd_topic_{i}' for i in range(15)]
df_svd = pd.DataFrame(X_svd, columns=svd_cols, index=df.index)

In [14]:
from textblob import TextBlob

# Calculate polarity (-1 to +1) and subjectivity (0 to 1)
#df['sentiment_polarity'] = df['lyrics'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
#df['sentiment_subjectivity'] = df['lyrics'].apply(lambda x: TextBlob(str(x)).sentiment.subjectivity)

In [75]:
df.head()

,mean_syllables_word,mean_words_sentence,n_sentences,n_words,sentence_similarity,track_id,vocabulary_wealth,popularity,lyrics,word_density,lang,is_english,sentiment_polarity,ttr,repetition_score,avg_word_length,syllables_per_line,syllables_per_word,syllable_variation,novel_word_prop
0,1.10,5.65,31,326,0.043011,13keyz9ikBe6ZpRasw7l4X,0.45,52.0,It was one of those times what a real good tim...,10.516126,en,1,0.145833,0.296875,0.703125,4.134375,440.0,1.349693,0.0,0.0
1,1.37,4.77,74,532,0.050352,1WugzepXsLjnsM0K4UaWYc,0.59,55.0,Ain't' a lifestyle that I would rather Look in...,7.189188,en,1,0.057386,0.423828,0.576172,4.224609,738.0,1.387218,0.0,0.0
2,1.95,3.38,72,430,0.028560,2MO6oEAlMKcsfI8xP3yoy8,0.49,46.0,"Tú, tú eres lo que yo escogí Yo quiero perderm...",5.972221,es,0,0.000000,0.304651,0.695349,3.730233,688.0,1.600000,0.0,0.0
3,1.16,2.99,68,368,0.047849,1i4St7fmSUE9nB3R9n8fol,0.47,36.0,"Now I've had the time of my life No, I never f...",5.411764,en,1,0.029545,0.335277,0.664723,3.565598,463.0,1.258152,0.0,0.0
4,1.32,4.21,39,256,0.040486,3UyfvY3Gs6d4wvq8O4ANqQ,0.60,6.0,On est pas là pour payer les dettes On a tous ...,6.564101,fr,0,0.000000,0.459574,0.540426,4.127660,353.0,1.378906,0.0,0.0


In [78]:
track_ids = df["track_id"].astype(str).tolist()

# 2. Pass your local CSV file explicitly to Lucas's function
train_ids, val_ids, test_ids = map(
    set, 
    grouped_track_id_split(track_ids, metadata_csv="spotify_tracks.csv")
)

# 3. Filter your lyrics DataFrame into leak-free sets
train_df = df[df["track_id"].astype(str).isin(train_ids)].reset_index(drop=True)
val_df   = df[df["track_id"].astype(str).isin(val_ids)].reset_index(drop=True)
test_df  = df[df["track_id"].astype(str).isin(test_ids)].reset_index(drop=True)

print(f"Train tracks: {len(train_df)}")
print(f"Val tracks:   {len(val_df)}")
print(f"Test tracks:  {len(test_df)}")

fixed artist/album-isolated split train=55594 (70.0%), validation=11913 (15.0%), test=11913 (15.0%), fingerprint=27c148725c44eff8
Train tracks: 55594
Val tracks:   11913
Test tracks:  11913


In [83]:
clean_features = [
    'mean_syllables_word',
    'mean_words_sentence',
    'n_sentences',
    'n_words',
    'sentence_similarity',
    'vocabulary_wealth',
    'word_density',
    'is_english',
    'repetition_score',
    'avg_word_length',    
    'syllables_per_line',
    'syllables_per_word'
]

X_clean = df[clean_features]
y = df['popularity']

# 2. Get leak-free Grouped Track IDs (70% Train / 15% Val / 15% Test)
track_ids = df['track_id'].astype(str).tolist()
train_ids, val_ids, test_ids = map(
    set, 
    grouped_track_id_split(track_ids, metadata_csv="spotify_tracks.csv")
)

# 3. Create Train / Val / Test Masks
train_mask = df['track_id'].astype(str).isin(train_ids)
val_mask   = df['track_id'].astype(str).isin(val_ids)
test_mask  = df['track_id'].astype(str).isin(test_ids)

X_train, y_train = X_clean[train_mask], y[train_mask]
X_val,   y_val   = X_clean[val_mask],   y[val_mask]
X_test,  y_test  = X_clean[test_mask],  y[test_mask]

# 4. Scale features (Fit ONLY on X_train to prevent leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# 5. Initialize and fit Random Forest model
rf_clean = RandomForestRegressor(
    n_estimators=225, 
    random_state=10, 
    min_samples_split=5, 
    min_samples_leaf=2, 
    max_features='sqrt', 
    max_depth=None,
    n_jobs=-1
)

rf_clean.fit(X_train_scaled, y_train)

# 6. Predict on all splits
train_preds = rf_clean.predict(X_train_scaled)
print(f"Train R² Score : {r2_score(y_train, train_preds):.4f}")

fixed artist/album-isolated split train=55594 (70.0%), validation=11913 (15.0%), test=11913 (15.0%), fingerprint=27c148725c44eff8
Train R² Score : 0.7557


In [84]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 8, 12],            # Prevents trees from memorizing training data
    'min_samples_split': [10, 20, 50],     # Requires more samples before splitting
    'min_samples_leaf': [5, 10, 20],       # Smooths predictions at leaf nodes
    'max_features': ['sqrt', 0.5]          # Subsamples features per split
}

best_val_r2 = -float('inf')
best_params = None
best_model = None

# Manual Grid Search over Train/Val sets
for depth in param_grid['max_depth']:
    for split in param_grid['min_samples_split']:
        for leaf in param_grid['min_samples_leaf']:
            for mf in param_grid['max_features']:
                
                model = RandomForestRegressor(
                    n_estimators=150,
                    max_depth=depth,
                    min_samples_split=split,
                    min_samples_leaf=leaf,
                    max_features=mf,
                    random_state=10,
                    n_jobs=-1
                )
                
                # Fit ONLY on training data
                model.fit(X_train_scaled, y_train)
                
                # Evaluate ONLY on validation data
                val_preds = model.predict(X_val_scaled)
                val_r2 = r2_score(y_val, val_preds)
                
                if val_r2 > best_val_r2:
                    best_val_r2 = val_r2
                    best_params = {
                        'max_depth': depth,
                        'min_samples_split': split,
                        'min_samples_leaf': leaf,
                        'max_features': mf
                    }
                    best_model = model

print("=== Hyperparameter Tuning Complete ===")
print("Best Validation R²:", f"{best_val_r2:.4f}")
print("Best Parameters   :", best_params)

=== Hyperparameter Tuning Complete ===
Best Validation R²: -0.0800
Best Parameters   : {'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 0.5}


In [85]:
test_preds = best_model.predict(X_test_scaled)
train_preds = best_model.predict(X_train_scaled)

print("\n=== Final Test Performance ===")
print(f"Train R² : {r2_score(y_train, train_preds):.4f}")
print(f"Val R²   : {best_val_r2:.4f}")
print(f"Test R²  : {r2_score(y_test, test_preds):.4f}")


=== Final Test Performance ===
Train R² : 0.4276
Val R²   : -0.0800
Test R²  : -0.0943


grid search random search


=== Random Forest Performance ===      
'mean_syllables_word',
    'mean_words_sentence',
    'n_sentences',
    'n_words',
    'sentence_similarity',
    'vocabulary_wealth'
RMSE (Root Mean Squared Error): 13.2630
MAE  (Mean Absolute Error):    9.9282
R² Score:                       0.4111


r2: 0.4153 250 n-estimators + word_density


In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Scatter plot with regression trendline (non-linear scatter distribution)
sns.regplot(
    data=clean_df, 
    x='n_words', 
    y='popularity',
    scatter_kws={'alpha': 0.4, 'color': '#1DB954'},  # Spotify Green with opacity
    line_kws={'color': 'darkred', 'linewidth': 2}      # Trendline
)

# Set labels and title
plt.title('Song Word Count (n_words) vs. Spotify Popularity Score', fontsize=14, pad=15)
plt.xlabel('Number of Words in Song (n_words)', fontsize=12)
plt.ylabel('Spotify Popularity Score (0-100)', fontsize=12)
plt.xlim(0, clean_df['n_words'].quantile(0.99))  # Cuts off extreme outliers for a cleaner view

plt.tight_layout()
plt.show()